In [0]:
from pyspark.sql.functions import col, desc, explode
from graphframes import GraphFrame

edge_csv_path = "dbfs:/FileStore/tables/lastfm_asia_edges.csv"
vert_csv_path = "dbfs:/FileStore/tables/lastfm_asia_target.csv"

edges_df_1 = spark.read.csv(edge_csv_path, header=True, inferSchema=True).selectExpr("node_1 as src", "node_2 as dst")
edges_df_2 = spark.read.csv(edge_csv_path, header=True, inferSchema=True).selectExpr("node_1 as dst", "node_2 as src")

edges_df = edges_df_1.unionByName(edges_df_2)

vert_df = spark.read.csv(vert_csv_path, header=True, inferSchema=True)

# edges_df.show()
# vert_df.show()

g = GraphFrame(vert_df, edges_df)
print(g.vertices.count())
print(g.edges.count())

7624
55612


In [0]:
top5outdegrees = g.outDegrees.orderBy("outDegree", ascending=False).limit(5)
top5outdegrees.show()

+----+---------+
|  id|outDegree|
+----+---------+
|7237|      216|
|3530|      175|
|4785|      174|
| 524|      172|
|3450|      159|
+----+---------+



In [0]:
top5indegrees = g.inDegrees.orderBy("inDegree", ascending=False).limit(5)
top5indegrees.show()

# Note indegrees and outdegrees are the same because edges are undirected.

+----+--------+
|  id|inDegree|
+----+--------+
|7237|     216|
|3530|     175|
|4785|     174|
| 524|     172|
|3450|     159|
+----+--------+



In [0]:
pagerank =  g.pageRank(resetProbability=0.15, tol=0.01)
pagerank.vertices.select("id", "pagerank").orderBy("pagerank", ascending=False).limit(5).show()

+----+------------------+
|  id|          pagerank|
+----+------------------+
|4811| 24.53196738920164|
|4785| 23.10128669710193|
|3530| 20.03970529156629|
|7237|18.991970442256143|
|3450| 17.76169836596356|
+----+------------------+



In [0]:
sc.setCheckpointDir("/tmp/spark-checkpoints")

connectedComponents = g.connectedComponents().select("id", "component").groupBy("component").count().orderBy("count", ascending=False).limit(5)
connectedComponents.show()

# Note this returns 1 component with id 0 that has 7624 vertices, which is ALL THE VERTICES IN THE GRAPH. So, this graph has ONLY 1 connected component, which is the whole graph.

+---------+-----+
|component|count|
+---------+-----+
|        0| 7624|
+---------+-----+



In [0]:
triangleCount = g.triangleCount().select("id", "count").orderBy("count", ascending=False).limit(5)
triangleCount.show()

+----+-----+
|  id|count|
+----+-----+
|7237| 1669|
| 524| 1117|
|3240| 1042|
|3597|  978|
| 763|  919|
+----+-----+

